In [ ]:
import h5py
import scipy.io
import numpy as np
import pickle
import gc
import os
from pathlib import Path

NR_DIR  = Path('NR_files')
TSR_DIR = Path('TSR_files')
SR_DIR  = Path('SR_files')

for d in [NR_DIR, TSR_DIR, SR_DIR]:
    mats = list(d.glob('*.mat')) if d.exists() else []
    print(f'{d.name}: {len(mats)} .mat files')

In [ ]:
def h5_decode_str(f, ref):
    try:
        return ''.join(chr(int(c)) for c in f[ref][:].flatten() if c != 0)
    except:
        return ''

def h5_extract_sentence(mat_path, subject_id, condition_id):
    rows = []
    with h5py.File(mat_path, 'r') as f:
        sd = f['sentenceData']
        n  = sd.shape[1] if len(sd.shape) > 1 else len(sd)
        for i in range(n):
            try:
                sref = sd[0, i] if len(sd.shape) > 1 else sd[i]
                sobj = f[sref]
                sent = h5_decode_str(f, sobj['content'][0, 0])
                if not sent.strip():
                    continue

                # Raw EEG — full 105-channel sentence-level
                try:
                    raw_ref = sobj['rawData'][0, 0]
                    raw_eeg = f[raw_ref][:].astype(np.float32)
                except:
                    raw_eeg = None

                # Word-level data
                wa = sobj['word']
                nw = wa.shape[1] if len(wa.shape) > 1 else len(wa)
                words_data = []
                for wi in range(nw):
                    try:
                        wref = wa[0, wi] if len(wa.shape) > 1 else wa[wi]
                        wobj = f[wref]
                        word_text = h5_decode_str(f, wobj['content'][0, 0])

                        # fixedLenthEEG — 256 timestep word EEG
                        try:
                            eeg_ref = wobj['fixedLenthEEG'][0, 0]
                            w_eeg   = f[eeg_ref][:].astype(np.float32)
                        except:
                            w_eeg = None

                        # Eye tracking
                        eye = {}
                        for key in ['nFixations','FFD','TRT','GD','GPT','SFD','meanPupilSize']:
                            try:
                                v = wobj[key][0, 0]
                                eye[key] = float(f[v][0,0]) if isinstance(v, h5py.Reference) else float(v)
                            except:
                                eye[key] = 0.0

                        # Spectral bands
                        spec = {}
                        for band in ['theta','alpha1','alpha2','beta1','beta2','gamma1','gamma2']:
                            try:
                                bref = wobj[band][0, 0]
                                spec[band] = f[bref][:].astype(np.float32).flatten()
                            except:
                                spec[band] = np.zeros(8, dtype=np.float32)

                        words_data.append({
                            'word'   : word_text,
                            'eeg'    : w_eeg,
                            'eye'    : eye,
                            'spec'   : spec,
                        })
                    except:
                        continue

                rows.append({
                    'subject_id'   : subject_id,
                    'condition'    : condition_id,
                    'sentence_idx' : i,
                    'sentence'     : sent.strip(),
                    'raw_eeg'      : raw_eeg,
                    'words'        : words_data,
                })
            except:
                continue
    return rows

print('✅ h5py extractor ready')

In [ ]:
def _str(raw):
    try:
        if isinstance(raw, str): return raw.strip()
        return str(np.array(raw).flatten()[0]).strip()
    except:
        return ''

def scipy_extract_sentence(mat_path, subject_id, condition_id):
    rows = []
    mat  = scipy.io.loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    sd   = mat['sentenceData']
    if not hasattr(sd, '__len__'):
        sd = [sd]

    for i, sent in enumerate(sd):
        try:
            sentence_text = _str(sent.content)
            if not sentence_text.strip():
                continue

            # Raw sentence EEG
            try:
                raw_eeg = np.array(sent.rawData).astype(np.float32)
            except:
                raw_eeg = None

            words = sent.word
            if not hasattr(words, '__len__'):
                words = [words]

            words_data = []
            for word in words:
                try:
                    word_text = _str(word.content)

                    # fixedLenthEEG
                    try:
                        w_eeg = np.array(word.fixedLenthEEG).astype(np.float32)
                    except:
                        w_eeg = None

                    # Eye tracking
                    eye = {}
                    for key in ['nFixations','FFD','TRT','GD','GPT','SFD','meanPupilSize']:
                        try:
                            eye[key] = float(np.array(getattr(word, key)).flatten()[0])
                        except:
                            eye[key] = 0.0

                    # Spectral bands
                    spec = {}
                    for band in ['theta','alpha1','alpha2','beta1','beta2','gamma1','gamma2']:
                        try:
                            spec[band] = np.array(getattr(word, band)).astype(np.float32).flatten()
                        except:
                            spec[band] = np.zeros(8, dtype=np.float32)

                    words_data.append({
                        'word'  : word_text,
                        'eeg'   : w_eeg,
                        'eye'   : eye,
                        'spec'  : spec,
                    })
                except:
                    continue

            rows.append({
                'subject_id'   : subject_id,
                'condition'    : condition_id,
                'sentence_idx' : i,
                'sentence'     : sentence_text.strip(),
                'raw_eeg'      : raw_eeg,
                'words'        : words_data,
            })
        except:
            continue
    return rows

print('✅ scipy extractor ready')

In [ ]:
def load_condition(folder, suffix, condition_id):
    """
    Load all subjects from a folder.
    Auto-detects Z vs Y prefix and calls correct extractor.
    suffix: 'NR', 'TSR', or 'SR'
    """
    all_rows = []
    mat_files = sorted(Path(folder).glob('*.mat'))
    print(f'\nLoading {suffix} ({len(mat_files)} files)...')

    for mat_path in mat_files:
        # Extract subject ID from filename e.g. resultsZAB_NR.mat → ZAB
        name = mat_path.stem   # e.g. resultsZAB_NR
        try:
            subject_id = name.replace('results','').replace(f'_{suffix}','').strip()
        except:
            subject_id = name

        print(f'  {subject_id}...', end=' ', flush=True)
        try:
            if subject_id.startswith('Z'):
                rows = scipy_extract_sentence(str(mat_path), subject_id, condition_id)
            else:
                rows = h5_extract_sentence(str(mat_path), subject_id, condition_id)
            print(f'{len(rows)} sentences')
            all_rows.extend(rows)
            del rows
        except Exception as e:
            print(f'ERROR: {e}')
        gc.collect()

    return all_rows


# ── NR ────────────────────────────────────────────────────────
nr_rows = load_condition(NR_DIR, 'NR', condition_id=0)
with open('NR_data.pkl', 'wb') as f:
    pickle.dump(nr_rows, f, protocol=4)
print(f'✅ NR_data.pkl saved — {len(nr_rows)} rows  ({os.path.getsize("NR_data.pkl")/1e6:.0f} MB)')
del nr_rows; gc.collect()

# ── TSR ───────────────────────────────────────────────────────
tsr_rows = load_condition(TSR_DIR, 'TSR', condition_id=1)
with open('TSR_data.pkl', 'wb') as f:
    pickle.dump(tsr_rows, f, protocol=4)
print(f'✅ TSR_data.pkl saved — {len(tsr_rows)} rows  ({os.path.getsize("TSR_data.pkl")/1e6:.0f} MB)')
del tsr_rows; gc.collect()

# ── SR ────────────────────────────────────────────────────────
sr_rows = load_condition(SR_DIR, 'SR', condition_id=2)
with open('SR_data.pkl', 'wb') as f:
    pickle.dump(sr_rows, f, protocol=4)
print(f'✅ SR_data.pkl saved — {len(sr_rows)} rows  ({os.path.getsize("SR_data.pkl")/1e6:.0f} MB)')
del sr_rows; gc.collect()

print('\n✅ Done. Files saved: NR_data.pkl / TSR_data.pkl / SR_data.pkl')